In [1]:
import pandas as pd
import numpy as np

In [2]:
events = pd.read_parquet(f"data/events.parquet")
races = pd.read_parquet(f"data/races.parquet")
runners = pd.read_parquet(f"data/runners.parquet")

In [3]:
fbase = "output/data"

# Pre-prep data

In [4]:
lmap = {
    "Aint": "Aintree",
    "Bang": "Bangor On Dee",
    "Bangor": "Bangor On Dee",
    "Bangor on Dee": "Bangor On Dee",
    "Bev": "Beverly",
    "Brig": "Brighton",
    "Carl": "Carlisle",
    "Cart": "Cartmel",
    "Catt": "Cattertick",
    "ChelmC": "Chelmsford City",
    "ChelmC PA": "Chelmsford City",
    "Chelt": "Cheltenham",
    "Chep": "Chepstow",
    "Chest": "Chester",
    "Donc": "Doncaster",
    "Epsm": "Epsom",
    "Extr": "Exeter",
    "Fake": "Fakenham",
    "FfosL": "Ffoss Las",
    "Font": "Fontwell",
    "GLghs": "Chelmsford City",
    "Good": "Goodwood",
    "Ham": "Hamilton",
    "Hayd": "Haydock",
    "Here": "Hereford",
    "Hex": "Hexham",
    "Hunt": "Huntingdon",
    "Kemp": "Kempton",
    "Leic": "Leicester",
    "Ling": "Lingfield",
    "Lud": "Ludlow",
    "MrktR": "Market Rasen",
    "Muss": "Musselburgh",
    "Newb": "Newbury",
    "Newc": "Newcastle",
    "Newm": "Newmarket",
    "Newt": "Newton Abbot",
    "Nott": "Nottingham",
    "Plump": "Plumpton",
    "Ponte": "Pontefract",
    "Redc": "Redcar",
    "Salis": "Salisbury",
    "Sand": "Sandown",
    "Sedge": "Sedgefield",
    "Sthl": "Southwell",
    "Strat": "Stratford",
    "Taun": "Taunton",
    "Uttox": "Uttoxeter",
    "Warw": "Warwick",
    "Weth": "Wetherby",
    "Winc": "Wincanton",
    "Wind": "Windsor",
    "Wolv": "Wolverhampton",
    "Worc": "Worcester",
    "Yarm": "Yarmouth"
}
events.location = events.location.replace(lmap)

In [5]:
runners["returns"] = runners.bsp * runners.win_lose

# Favourite-Longshot Bias

In [6]:
bsp_bins=[1, 2, 3, 4, 6, 10, 20, 30, 50, 100, 1001]

In [7]:
ticks = np.concat((
    np.arange(1, 2, 0.01),
    np.arange(2,3,0.02),
    np.arange(3,4,0.05),
    np.arange(4,6,0.1),
    np.arange(6,10,0.2),
    np.arange(10,20,0.5),
    np.arange(20,30,1),
    np.arange(30,50,2),
    np.arange(50,100,5),
    np.arange(100,1000,10),
    [1001]
))

## General

In [8]:
g = runners.groupby(pd.cut(runners.bsp, bins=ticks))

In [9]:
df = pd.DataFrame({"mean": g.win_lose.mean(), "count": g.size(), "volume": g.pptradedvol.sum()})

In [10]:
df.index=df.index.categories.right

In [11]:
df.to_csv(f"{fbase}/all-fl.csv")

## Groups

per race stats

In [12]:
rg = runners.groupby("race_id")

nraces = races.assign(
    ppvol=rg.pptradedvol.sum(),
    nrunners=rg.size(),
    overround=rg.implied_odds.sum()
)

add groups that are needed

In [13]:
df = runners.join(nraces, on="race_id").join(events, on="event_id")

In [14]:
cats = {
    "race_no": ("race_no", [1,2,3,4,5,6,7,8]),
    "distance_f": ("distance_f", df.distance_f.value_counts().head(10).index),
    "hour": (df.race_time.dt.hour, np.arange(12,20,1)),
    "race_type": ("race_type", df.race_type.value_counts().head(10).index),
    "location": ("location", df.location.value_counts().head(10).index),
    "nrunners": ("nrunners", df.nrunners.value_counts().head(10).index),
    "ppvol": (pd.qcut(df.ppvol, q=10), slice(None)),
    "overround": (pd.qcut(df.overround, q=10), slice(None))
}

In [15]:
for c in cats:
    for a, v in [("mean", "win_lose"), ("count", "win_lose"), ("sum", "pptradedvol")]:
        pt = df.pivot_table(
            index=pd.cut(df.bsp, bins=ticks),
            columns=cats[c][0],
            values=v,
            aggfunc=a
        ).loc[:, cats[c][1]]
        pt = pt.reindex(sorted(pt.columns), axis=1)
        pt.columns = pt.columns.map(str).str.replace(",", ";")
        pt.index=pt.index.categories.right
        pt.to_csv(f"{fbase}/{c}-fl-{a}.csv")

In [16]:
pt = df.pivot_table(
    index=pd.cut(df.bsp, bins=ticks),
    columns="race_no",
    values=["win_lose", "pptradedvol", "win_lose"],
    aggfunc=["mean", "sum", "count"]
)
pt

mean                                              \
                  pptradedvol                                               
race_no                    1              2             3              4    
bsp                                                                         
(1.0, 1.01]      1.150880e+06            NaN           NaN            NaN   
(1.01, 1.02]     7.204030e+05            NaN           NaN            NaN   
(1.02, 1.03]     6.632248e+05            NaN  1.093286e+06            NaN   
(1.03, 1.04]     7.900992e+05  453130.312500  1.081288e+06  876129.687500   
(1.04, 1.05]     4.072085e+05  741537.312500  9.756135e+05  633300.875000   
...                       ...            ...           ...            ...   
(950.0, 960.0]   7.419796e+02     653.691223  8.329005e+02     828.010559   
(960.0, 970.0]   7.919971e+02     827.404480  7.953253e+02    1246.476074   
(970.0, 980.0]   7.741734e+02     743.946106  7.761748e+02     908.961548   
(980.0, 990.0]   7.307676e+02     696.514282  8.456378e+02     742.689575   
(990.0, 1001.0]  4.975919e+02     521.254395  5.491370e+02     632.143982   

                                                                        \
                                                                         
race_no                    5             6             7            8    
bsp                                                                      
(1.0, 1.01]      3.412257e+05           NaN           NaN          NaN   
(1.01, 1.02]     1.441296e+06           NaN  1.279604e+06          NaN   
(1.02, 1.03]              NaN           NaN  1.721552e+06          NaN   
(1.03, 1.04]     2.826830e+05  6.246374e+05           NaN          NaN   
(1.04, 1.05]     8.581126e+05  1.493605e+06  8.537292e+05          NaN   
...                       ...           ...           ...          ...   
(950.0, 960.0]   7.595900e+02  1.296463e+03  9.232817e+02   695.859985   
(960.0, 970.0]   8.072025e+02  1.117479e+03  8.186046e+02   590.065002   
(970.0, 980.0]   1.187279e+03  8.231191e+02  7.945743e+02  1155.369995   
(980.0, 990.0]   8.309021e+02  1.142434e+03  7.829806e+02   710.661987   
(990.0, 1001.0]  6.258256e+02  6.743716e+02  5.888418e+02   556.723267   

                                 ...    count                                  \
                                 ... win_lose                                   
race_no                  9   10  ...       6      7      8     9   10  11  12   
bsp                              ...                                            
(1.0, 1.01]             NaN NaN  ...      NaN    NaN    NaN   NaN NaN NaN NaN   
(1.01, 1.02]            NaN NaN  ...      NaN    1.0    NaN   NaN NaN NaN NaN   
(1.02, 1.03]            NaN NaN  ...      NaN    1.0    NaN   NaN NaN NaN NaN   
(1.03, 1.04]            NaN NaN  ...      1.0    NaN    NaN   NaN NaN NaN NaN   
(1.04, 1.05]            NaN NaN  ...      1.0    2.0    NaN   NaN NaN NaN NaN   
...                     ...  ..  ...      ...    ...    ...   ...  ..  ..  ..   
(950.0, 960.0]          NaN NaN  ...     27.0   12.0    5.0   NaN NaN NaN NaN   
(960.0, 970.0]          NaN NaN  ...     18.0   13.0    2.0   NaN NaN NaN NaN   
(970.0, 980.0]          NaN NaN  ...     22.0   16.0    2.0   NaN NaN NaN NaN   
(980.0, 990.0]          NaN NaN  ...     27.0   17.0    5.0   NaN NaN NaN NaN   
(990.0, 1001.0]  988.600952 NaN  ...   1243.0  905.0  148.0  10.0 NaN NaN NaN   

                             
                             
race_no          13  14  15  
bsp                          
(1.0, 1.01]     NaN NaN NaN  
(1.01, 1.02]    NaN NaN NaN  
(1.02, 1.03]    NaN NaN NaN  
(1.03, 1.04]    NaN NaN NaN  
(1.04, 1.05]    NaN NaN NaN  
...              ..  ..  ..  
(950.0, 960.0]  NaN NaN NaN  
(960.0, 970.0]  NaN NaN NaN  
(970.0, 980.0]  NaN NaN NaN  
(980.0, 990.0]  NaN NaN NaN  
(990.0, 1001.0] NaN NaN NaN  

[350 rows x 225 columns]